# Testing Nethack wiki RAG
The wiki is not the same as the notebook. It has a lot more info in it

In [1]:
import xml.etree.ElementTree as ET
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pickle
import os

/homes/53/fpinto/BALROG/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Trying out Functions

In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2") # embedding model

# Paths for persistence
FAISS_INDEX_PATH = "/homes/53/fpinto/BALROG/balrog/docs_for_rag/nethack_wiki_faiss.index"
STORE_PATH = "/homes/53/fpinto/BALROG/balrog/docs_for_rag/nethack_wiki_store.pkl"

In [3]:
def parse_mediawiki_xml(xml_path):
    """Parses MediaWiki XML and extracts full text per page."""
    ns = {"mw": "http://www.mediawiki.org/xml/export-0.10/"}
    tree = ET.parse(xml_path)
    root = tree.getroot()

    pages = root.findall(".//mw:page", namespaces=ns)
    extracted = []

    for page in pages:
        title = page.find("mw:title", namespaces=ns).text
        revision = page.find(".//mw:revision/mw:text", namespaces=ns)

        if revision is not None and revision.text:
            text_content = revision.text.strip()
            extracted.append((title, text_content))

    # print(f"Extracted {len(extracted)} pages.")
    return extracted


def build_faiss_index(data, model, faiss_index_path=None, storage_path=None):
    """Encodes pages with embeddings and stores in FAISS HNSW index along with full text."""
    titles, texts = zip(*data)  # Extract titles and content
    
    # Convert text into embeddings
    embeddings = model.encode(texts, convert_to_numpy=True)
    faiss.normalize_L2(embeddings)  # Normalize for cosine similarity

    # Use HNSW for scalable nearest-neighbor search
    dim = embeddings.shape[1]
    index = faiss.IndexHNSWFlat(dim, 32)  # 32 neighbors in HNSW graph
    index.hnsw.efConstruction = 128  # Better recall
    index.add(embeddings)

    # Store full content alongside titles
    doc_store = [{"title": t, "content": c} for t, c in zip(titles, texts)]

    # Save FAISS index and document store
    faiss.write_index(index, faiss_index_path)
    with open(storage_path, "wb") as f:
        pickle.dump(doc_store, f)

    print(f"FAISS index and document store saved.")
    return index, doc_store


def load_faiss_index(faiss_index_path, storage_path):
    """Loads the FAISS index and document store if they exist."""
    if os.path.exists(faiss_index_path) and os.path.exists(storage_path):
        index = faiss.read_index(faiss_index_path)
        with open(storage_path, "rb") as f:
            doc_store = pickle.load(f)  # Load stored titles and texts
        print("Loaded FAISS index and document store from disk.")
        return index, doc_store
    else:
        print("No saved index found. Build it first.")
        return None, None
    

def search_faiss(index, doc_store, query, model, top_k=5):
    """Search FAISS index for similar documents and return titles + content."""
    query_embedding = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)  # Normalize query

    distances, indices = index.search(query_embedding, top_k)

    # Retrieve the titles and content for top-k results
    results = []
    for idx in indices[0]:
        doc = doc_store[idx]  # Retrieve from dictionary
        results.append((doc["title"], doc["content"]))

    return results

In [11]:
xml_file = "/homes/53/fpinto/BALROG/balrog/docs_for_rag/nethackwiki.xml"

# Load existing FAISS index or create a new one
faiss_index, page_titles = load_faiss_index(FAISS_INDEX_PATH, STORE_PATH)
if faiss_index is None:
    parsed_data = parse_mediawiki_xml(xml_file)
    faiss_index, page_titles = build_faiss_index(parsed_data, model=model, faiss_index_path=FAISS_INDEX_PATH, storage_path=STORE_PATH)

# Search query
query_text = """Weapons
a - a blessed +1 quarterstaff (weapon in hands)
"""
search_faiss(faiss_index, page_titles, query_text, model=model)

Loaded FAISS index and document store from disk.


[('Quarterstaff',
  '{{languages}}\n{{weapon\n  |color=wood color\n  |tile=[[image:quarterstaff.png]]\n  |name=quarterstaff\n  |appearance=staff\n  |smalldmg=1d6\n  |largedmg=1d6\n  |size=two-handed\n  |cost=5\n  |weight=40\n  |material=wood\n}}\n\nA \'\'\'quarterstaff\'\'\' (plural \'\'\'quarterstaffs\'\'\' or \'\'\'quarterstaves\'\'\') is a type of [[weapon]] that appears in \'\'[[NetHack]]\'\'. It is [[two-handed]] and made of [[wood]], and simply appears as a [[staff]] when unidentified.\n\nThe quarterstaff is the [[base item]] type for the [[artifact]] [[The Staff of Aesculapius]].\n\n==Generation==\nAll [[Wizard]]s start with a [[blessed]] +1 quarterstaff.{{refsrc|src/u_init.c|162|version=NetHack 3.6.7}}\n\nQuarterstaves make up about 1.1% of randomly generated weapons (on the floor, as [[death drop]]s, or in [[shop]]s). [[Arch-lich]]es have a {{frac|2|9}} chance of being generated with a quarterstaff, but lack any weapon attacks and so will not use it in combat.{{refsrc|src/make

## Turn this into a class

In [3]:
class NethackWikiSearch:
    """Handles parsing, indexing, and searching MediaWiki XML dumps with FAISS."""
    
    def __init__(self, wiki_path="/homes/53/fpinto/BALROG/balrog/docs_for_rag/nethackwiki.xml", faiss_index_path="/homes/53/fpinto/BALROG/balrog/docs_for_rag/nethack_wiki_faiss.index", 
                 storage_path="/homes/53/fpinto/BALROG/balrog/docs_for_rag/nethack_wiki_store.pkl", model_name="all-MiniLM-L6-v2"): # Change this to config in the main code
        self.model = SentenceTransformer(model_name)
        self.wiki_path = wiki_path
        self.faiss_index_path = faiss_index_path
        self.storage_path = storage_path
        self.index = None
        self.doc_store = None
        self.top_k = 5

    def __parse_xml(self):
        """Parses MediaWiki XML and extracts full text per page."""
        ns = {"mw": "http://www.mediawiki.org/xml/export-0.10/"}
        tree = ET.parse(self.wiki_path)
        root = tree.getroot()

        pages = root.findall(".//mw:page", namespaces=ns)
        extracted = []

        for page in pages:
            title = page.find("mw:title", namespaces=ns).text
            revision = page.find(".//mw:revision/mw:text", namespaces=ns)

            if revision is not None and revision.text:
                text_content = revision.text.strip()
                extracted.append((title, text_content))

        return extracted
    

    def __build_index(self):
        """Encodes pages with embeddings and stores in FAISS HNSW index along with full text."""
        data = self.__parse_xml()
        titles, texts = zip(*data)  # Extract titles and content
        
        # Convert text into embeddings
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        faiss.normalize_L2(embeddings)  # Normalize for cosine similarity

        # Use HNSW for scalable nearest-neighbor search
        dim = embeddings.shape[1]
        self.index = faiss.IndexHNSWFlat(dim, 32)
        self.index.hnsw.efConstruction = 128  # Better recall
        self.index.add(embeddings)

        # Store full content alongside titles
        self.doc_store = [{"title": t, "content": c} for t, c in zip(titles, texts)]

        # Save FAISS index and document store
        faiss.write_index(self.index, self.faiss_index_path)
        with open(self.storage_path, "wb") as f:
            pickle.dump(self.doc_store, f)

        print("FAISS index and document store saved.")


    def load_index(self):
        """Loads the FAISS index and document store if they exist."""
        if not (os.path.exists(self.faiss_index_path) and os.path.exists(self.storage_path)):
            print("No saved index found. Building the index.")
            self.__build_index()

        self.index = faiss.read_index(self.faiss_index_path)
        with open(self.storage_path, "rb") as f:
            self.doc_store = pickle.load(f)

        return print("Loaded FAISS index and document store from disk.")


    def search(self, query):
        """Search FAISS index for similar documents and return titles + content."""
        if self.index is None or self.doc_store is None:
            print("Index not loaded. Load or build it first.")
            return []

        query_embedding = self.model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(query_embedding)  # Normalize query

        distances, indices = self.index.search(query_embedding, self.top_k)

        return [(self.doc_store[idx]["title"], self.doc_store[idx]["content"]) for idx in indices[0]]


In [4]:
retriever = NethackWikiSearch()
retriever.load_index()

Loaded FAISS index and document store from disk.


In [5]:
retriever.search("""potions""")

[('Potion of ESP',
  "{{potion|name=ESP|cost=150}}\n\nA '''potion of ESP''' is a [[potion]] introduced in [[SLASH'EM]]. It grants [[intrinsic]] [[telepathy]] temporarily if uncursed and permanently if blessed. If cursed, removes intrinsic telepathy.\n\n==Strategy==\n\nA character is by no means guaranteed to find a [[floating eye]] corpse; a blessed potion of ESP can be a useful way to gain the intrinsic for such unlucky ones.  They are also useful for non-chaotic characters wishing to commit [[murder]] who lack a [[tinning kit]].\n\nFor players who already possess telepathy and don't anticipate losing it, these potions are a good candidate for dilution.\n\n[[Category:SLASH'EM]]\n[[Category:SLASH'EM items]]\n[[Category:Potions]]"),
 ('Potion of wonder',
  "{{potion\n |name=wonder\n |cost=200\n}}\n\nA '''potion of wonder''' is a type of [[potion]] that appears in [[FIQHack]]. It can confer a variety of effects when [[quaff]]ed or [[dipped]] into.\n\n==Description== \nWhen quaffed, a non